# Loading Data from a probe 
Here we show how to load the data for the specific pretrained probe

In [ ]:
from omegaconf import OmegaConf
from pathlib import Path
import sys
from hydra import initialize, compose
import os


from data_handler import DataHandler
# 1) Make your repo importable (so `from misc.probe_data import ProbeData` works)
def load_hydra_config_with_params(model, datapack, probe, config_name):
    with initialize(version_base="1.1", config_path="./configs"):
        cfg = compose(config_name=config_name, overrides=[f"model={model}", f"datapack={datapack}", f"probe={probe}"])
    OmegaConf.set_struct(cfg, False)  # Allow overriding
    trial_name = cfg.trial_name
    if cfg.probe['name'] == 'mean_diff':
        cfg.search = False
    if cfg.search:
        trial_name += "_search"
    trial_name += f'_task-{cfg.task}'
    cfg["trial_name"] = trial_name
    # if cfg["task"] == 2:
    #     cfg["probe"]["assume_known_positives"] = False
    cfg["output_dir"] = os.path.join(cfg.output_dir, trial_name)
    OmegaConf.set_struct(cfg, True)
    return OmegaConf.to_container(cfg, resolve=True)

# 1. Load Data and Test Bags

In [ ]:
datapack_name = 'city_locations'
model_name = 'llama-3-8b'
probe_name = 'sawmil'
cfg = load_hydra_config_with_params(model = model_name, datapack=datapack_name, probe = probe_name, config_name="probe_mil")
datapack_params = cfg["datapack"]
dh = DataHandler(model=model_name,datasets=datapack_params['datasets'], dataset_path='./datasets/', activation_type=datapack_params["agg"], with_calibration=True, load_scores='default', verbose=True)
dh.assemble(exclusive_split=datapack_params["exclusive_split"], test_size=datapack_params["test_size"], calibration_size=datapack_params["cal_size"], seed=datapack_params["random_seed"])
dh.get_train_df().shape, dh.get_test_df().shape,  dh.get_cal_df().shape, dh.get_dataframe().shape

In [ ]:
layer_id = 13 # we are going to work with the activations from the layer 13
bags = dh.test_bags(layer_id)['embeddings']
train_bags = dh.train_bags(layer_id)['embeddings']
train_y = dh.get_train_labels()

# 2. Load `sAwMIL` Probe data (for True, False and Neither Directions)

In [ ]:
from misc.probe_data import ProbeData
from probes.multiclass import OvAProjector, OvAScoresSkWrapper, MulticlassProbe

# Here we load the one-vs-all probes that we have trained
# Per-class readers for OvA:
is_false_data = ProbeData(f"/outputs/probes/{probe_name}/{model_name}/{datapack_name}_search_task-1/")
is_true_data = ProbeData(f"/outputs/probes/{probe_name}/{model_name}/{datapack_name}_search_task-0/")
is_neither_data = ProbeData(f"/outputs/probes/{probe_name}/{model_name}/{datapack_name}_search_task-2/")

projector = OvAProjector({0: is_false_data, 1: is_true_data, 2: is_neither_data}, layer_id=layer_id)
wrapped = OvAScoresSkWrapper(projector)


In [ ]:
# Last bag in out `City Locations` dataset is correct (let's look at the scores)
projector.score_bags([bags[-1]])
# array([[ -6.0819335 ,   1.07445522, -18.83119148]])  -> the score for the is_true is high, that is what we want 
# but we need to tie these scores into a multiclass probe 

In [ ]:
# `wrapped` object allows us to use a transform method of the SKLEARN package (essentially it returns the same result)
wrapped.transform([bags[-1]])

# 3. Let's transform that into probabilities

In [ ]:
## Now we can create a multiclass probe
probe = MulticlassProbe(projector)
probe.fit(train_bags, train_y)

In [ ]:
# now we get class probabilities
probe.predict_proba([bags[-1]])
# array([[2.00342353e-03, 9.97993997e-01, 2.57963060e-06]])

# 4. Loading Information from the Multiclass Probes

In [ ]:
from misc.probe_data import MulticlassProbeData

mpd = MulticlassProbeData(f'outputs/probes/{probe_name}/{model_name}/{datapack_name}_search_task--1')
mpd.metrics_conformal(13)